# 3D Gaussian Splatting Pipeline 控制中枢

这是项目的**指令记录本和执行中心**，用于：
- ✅ 验证环境配置
- ✅ 加载官方演示数据
- ✅ 运行 3DGS 训练演示
- ✅ 执行完整 pipeline（COLMAP → 3DGS → Evaluation）
- ✅ 记录日志和保存结果

**最后更新**: 2026-03-22
**当前环境**: gaussian_splatting (PyTorch 2.1.2, CUDA 11.8)

## 1️⃣ 导入必要的库和模块

In [ ]:
import os
import sys
import subprocess
import json
from pathlib import Path
from datetime import datetime
import numpy as np
import torch

# 配置项目路径（使用相对路径，兼容团队协作）
# 假设 notebook 在 notebooks/ 目录，项目根目录在上一级
NOTEBOOK_DIR = Path(__file__).resolve().parent if '__file__' in dir() else Path.cwd()
if NOTEBOOK_DIR.name != 'notebooks':
    # 在 Jupyter 中运行，用相对路径查找
    PROJECT_ROOT = Path.cwd()
    if not (PROJECT_ROOT / '.git').exists():
        # 如果当前目录没有 .git，往上找
        PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR.parent

CONDA_HOME = Path.home() / "miniconda3"
ENV_NAME = "gaussian_splatting"
GS_DIR = PROJECT_ROOT / "third_party" / "gaussian-splatting"
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
LOG_DIR = PROJECT_ROOT / "logs"

# 创建输出目录
OUTPUT_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

# 配置日志
import logging
log_file = LOG_DIR / f"pipeline_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

## 2️⃣ 环境检查和验证

In [ ]:
print("="*60)
print("3DGS 环境检查")
print("="*60)

# 检查 PyTorch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 数量: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  - GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"CUDA 版本: {torch.version.cuda}")

# 检查核心依赖
print("\n核心依赖:")
try:
    import cv2
    print(f"✓ OpenCV {cv2.__version__}")
except:
    print("✗ OpenCV 未安装")

try:
    import numpy
    print(f"✓ NumPy {numpy.__version__}")
except:
    print("✗ NumPy 未安装")

try:
    from diff_gaussian_rasterization import GaussianRasterizer
    print(f"✓ diff_gaussian_rasterization 已编译")
except:
    print("✗ diff_gaussian_rasterization 未编译")

try:
    import colmap
    print(f"✓ COLMAP 可用")
except:
    print("⚠ COLMAP 模块未安装（可选）")

# 检查 3DGS 仓库
print("\n项目结构:")
print(f"✓ 项目根目录: {PROJECT_ROOT}")
print(f"✓ 3DGS 仓库: {GS_DIR}")
print(f"✓ 数据目录: {DATA_DIR}")
print(f"✓ 输出目录: {OUTPUT_DIR}")

print("\n环境检查完成 ✓")


## 3️⃣ 项目配置（Pipeline 参数）

在这里配置整个 pipeline 的关键参数：

In [ ]:
# =====================================================
# 🎛️ PIPELINE 配置参数
# =====================================================

# 数据集配置
config = {
    "dataset": {
        # 数据源选择: "minimal" | "custom" | "nerf_synthetic"
        # minimal: 极小演示数据（无需下载）
        # custom: 用户自定义数据
        # nerf_synthetic: NeRF Synthetic 数据集
        "source": "minimal",
        
        # 数据路径
        "path": str(DATA_DIR / "minimal_dataset"),
        
        # 图像目录
        "images_dir": "images",
        
        # 是否使用 COLMAP 预处理
        "use_colmap": False,
    },
    
    "colmap": {
        # COLMAP 工作目录
        "workspace": str(DATA_DIR / "colmap_workspace"),
        
        # COLMAP 相机内参（如果使用 NeRF Synthetic）
        "camera_model": "PINHOLE",
        "fx": 1111.11,
        "fy": 1111.11,
        "cx": 400,
        "cy": 400,
    },
    
    "3dgs": {
        # 3DGS 训练参数
        "iterations": 300,  # 快速演示用 300，完整训练用 30000
        "resolution": 1,    # 1=原分辨率, 2=1/2, 4=1/4 等
        "white_background": False,
        "sh_degree": 3,
        "position_lr_init": 0.00016,
        "position_lr_final": 0.0000016,
        "feature_lr": 0.0025,
        "opacity_lr": 0.05,
        "scaling_lr": 0.005,
        "rotation_lr": 0.001,
        
        # 输出配置
        "output_dir": str(OUTPUT_DIR / "3dgs_demo"),
        "save_iterations": [300],
        "test_iterations": [300],
    },
    
    "evaluation": {
        # 评估指标
        "compute_lpips": True,
        "compute_ssim": True,
        "compute_psnr": True,
    }
}

# 创建输出目录
Path(config["3dgs"]["output_dir"]).mkdir(parents=True, exist_ok=True)

logger.info(f"Pipeline 配置已加载 ✓")
logger.info(f"  Dataset: {config['dataset']['source']}")
logger.info(f"  3DGS iterations: {config['3dgs']['iterations']}")
logger.info(f"  Output dir: {config['3dgs']['output_dir']}")

print("✓ Pipeline 配置完成")
print(json.dumps(config, indent=2, ensure_ascii=False))


## 4️⃣ 执行 3DGS 训练演示（300 iterations）

**注意**: 
- 快速演示用 300 iterations（~2-5 分钟）
- 完整训练用 30000 iterations（~1 小时）
- 修改上面第 3 部分的 `config["3dgs"]["iterations"]` 来改变训练轮数

In [ ]:
def run_3dgs_training(config):
    """
    运行 3DGS 训练
    
    Args:
        config: 配置字典
    """
    logger.info("="*60)
    logger.info("启动 3DGS 训练")
    logger.info("="*60)
    
    # 切换到 3DGS 目录
    os.chdir(GS_DIR)
    
    # 建立训练命令
    cmd = [
        "python", "train.py",
        "-s", config["dataset"]["path"],
        "-m", config["3dgs"]["output_dir"],
        "--iterations", str(config["3dgs"]["iterations"]),
        "--resolution", str(config["3dgs"]["resolution"]),
        "--sh_degree", str(config["3dgs"]["sh_degree"]),
    ]
    
    # 如果需要白色背景
    if config["3dgs"]["white_background"]:
        cmd.append("-w")
    
    logger.info(f"训练命令: {' '.join(cmd)}")
    logger.info(f"开始时间: {datetime.now()}")
    
    try:
        # 运行训练
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=3600)
        
        if result.returncode == 0:
            logger.info("✓ 训练完成")
            logger.info(f"输出目录: {config['3dgs']['output_dir']}")
            return True
        else:
            logger.error(f"✗ 训练失败")
            logger.error(f"错误: {result.stderr}")
            return False
            
    except subprocess.TimeoutExpired:
        logger.error("✗ 训练超时（超过 1 小时）")
        return False
    except Exception as e:
        logger.error(f"✗ 训练中出错: {str(e)}")
        return False

# 执行训练
print("⏳ 启动 3DGS 训练...")
print(f"  数据路径: {config['dataset']['path']}")
print(f"  迭代次数: {config['3dgs']['iterations']}")
print(f"  输出目录: {config['3dgs']['output_dir']}")
print("\n  （这可能需要几分钟，请稍候...）")

success = run_3dgs_training(config)

if success:
    print("\n✓ 训练演示完成！")
    print(f"  - 模型存储在: {config['3dgs']['output_dir']}")
    print(f"  - PLY 点云文件已生成")
    print(f"  - 可在官方 viewer 中打开查看")
else:
    print("\n✗ 训练演示失败，查看日志以了解详情")

logger.info(f"完成时间: {datetime.now()}")


## 5️⃣ COLMAP 相机标定（可选）

如果使用自定义图像数据，需要先用 COLMAP 生成相机位姿和稀疏点云：

In [ ]:
# COLMAP 处理流程（仅当数据需要预处理时）
def run_colmap_preprocess(config):
    """
    使用 COLMAP 处理来自图像的相机位姿
    
    Args:
        config: 配置字典
    """
    if not config["dataset"]["use_colmap"]:
        logger.info("跳过 COLMAP 处理（已配置）")
        return True
    
    logger.info("="*60)
    logger.info("启动 COLMAP 相机标定")
    logger.info("="*60)
    
    colmap_script = PROJECT_ROOT / "scripts" / "reconstruction" / "run_colmap.sh"
    
    if not colmap_script.exists():
        logger.error(f"COLMAP 脚本不存在: {colmap_script}")
        return False
    
    try:
        cmd = [
            "bash", str(colmap_script),
            config["dataset"]["path"],
            config["colmap"]["workspace"]
        ]
        
        logger.info(f"执行命令: {' '.join(cmd)}")
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=1800)
        
        if result.returncode == 0:
            logger.info("✓ COLMAP 处理完成")
            return True
        else:
            logger.error(f"✗ COLMAP 处理失败: {result.stderr}")
            return False
            
    except Exception as e:
        logger.error(f"✗ COLMAP 出错: {str(e)}")
        return False

print("COLMAP 处理模块已加载（仅在 config['dataset']['use_colmap']=True 时激活）")
print(f"当前设置: use_colmap = {config['dataset']['use_colmap']}")


## 6️⃣ 训练结果评估

In [ ]:
import json
from pathlib import Path

def analyze_training_results(output_dir):
    """
    分析 3DGS 训练结果
    
    Args:
        output_dir: 输出目录
    """
    output_path = Path(output_dir)
    
    if not output_path.exists():
        print(f"✗ 输出目录不存在: {output_dir}")
        return
    
    print("="*60)
    print("训练结果分析")
    print("="*60)
    
    # 检查输出文件
    print("\n📁 生成的文件:")
    ply_files = list(output_path.glob("**/*.ply"))
    for ply_file in ply_files:
        size_mb = ply_file.stat().st_size / (1024*1024)
        print(f"  ✓ {ply_file.relative_to(output_path)} ({size_mb:.2f} MB)")
    
    # 检查指标文件
    results_file = output_path / "results.json"
    if results_file.exists():
        with open(results_file) as f:
            results = json.load(f)
        
        print("\n📊 训练指标:")
        if isinstance(results, dict):
            for key, value in results.items():
                if isinstance(value, (int, float)):
                    print(f"  - {key}: {value:.4f}")
    
    # 生成小结
    print("\n✅ 训练完成小结:")
    print(f"  - 输出目录: {output_dir}")
    print(f"  - PLY 点云文件: {len(ply_files)} 个")
    print(f"  - 可用于官方 reader 查看或转换为网页 viewer")
    
    logger.info(f"结果分析完成: {output_dir}")

# 执行评估（如果训练成功）
if success:
    print("\n分析训练结果...")
    analyze_training_results(config["3dgs"]["output_dir"])
else:
    print("\n⚠️ 训练未成功，跳过结果分析")


## 7️⃣ 完整 Pipeline 运行（一键执行）

点击下面的按钮，一次性执行完整的 pipeline（数据准备 → COLMAP → 3DGS 训练 → 评估）：

In [ ]:
def run_full_pipeline(config):
    """
    运行完整的 3DGS pipeline
    """
    print("\n" + "="*60)
    print("启动完整 3DGS Pipeline")
    print("="*60)
    
    start_time = datetime.now()
    logger.info(f"Pipeline 开始时间: {start_time}")
    
    # Step 1: 环境检查
    print("\n[1/3] 验证环境...")
    logger.info("Step 1: 环境检查 ✓")
    
    # Step 2: COLMAP（可选）
    if config["dataset"]["use_colmap"]:
        print("\n[2/3] 运行 COLMAP 相机标定...")
        logger.info("Step 2: COLMAP 开始")
        if not run_colmap_preprocess(config):
            logger.error("COLMAP 失败，中止 pipeline")
            return False
        logger.info("Step 2: COLMAP 完成 ✓")
    else:
        print("\n[2/3] 跳过 COLMAP（使用现成的 transforms.json）")
        logger.info("Step 2: COLMAP 跳过（配置不使用）")
    
    # Step 3: 3DGS 训练
    print("\n[3/3] 运行 3DGS 训练...")
    logger.info("Step 3: 3DGS 训练开始")
    if not run_3dgs_training(config):
        logger.error("3DGS 训练失败")
        return False
    logger.info("Step 3: 3DGS 训练完成 ✓")
    
    # 完成
    end_time = datetime.now()
    duration = (end_time - start_time).total_seconds() / 60
    
    print("\n" + "="*60)
    print("✅ Pipeline 完成！")
    print("="*60)
    logger.info(f"Pipeline 完成时间: {end_time}")
    logger.info(f"总耗时: {duration:.2f} 分钟")
    
    # 分析结果
    analyze_training_results(config["3dgs"]["output_dir"])
    
    return True

# 取消注释下面这行来执行完整 pipeline（不建议在演示时执行，会很耗时）
# run_full_pipeline(config)

print("✓ 完整 Pipeline 函数已定义")
print("使用: run_full_pipeline(config) 执行完整流程")


## 8️⃣ 日志和常用指令参考

In [ ]:
print("="*60)
print("📝 日志和调试信息")
print("="*60)

print(f"\n日志文件路径: {log_file}")
print("\n最近日志内容:")
print("-"*60)

# 显示最后 20 行日志
try:
    with open(log_file) as f:
        lines = f.readlines()
        for line in lines[-20:]:
            print(line.rstrip())
except:
    print("（暂无日志）")

print("\n" + "-"*60)
print("\n常用命令（在终端中执行）:")
print("-"*60)

commands = {
    "激活环境": "source $HOME/miniconda3/bin/activate gaussian_splatting",
    "检查环境": "bash scripts/quick_env_check.sh",
    "运行 3DGS 训练": "cd third_party/gaussian-splatting && python train.py -s <数据路径> -m <输出路径>",
    "查看训练日志": f"tail -f {log_file}",
    "列出所有输出": f"ls -la {OUTPUT_DIR}",
    "COLMAP 处理": "bash scripts/reconstruction/run_colmap.sh <图像目录> <工作目录>",
}

for desc, cmd in commands.items():
    print(f"\n{desc}:")
    print(f"  $ {cmd}")

print("\n" + "="*60)


## 📋 快速导航

### 🚀 Quick Start（首次使用）
1. **第 2️⃣ 部分** → 运行环境检查，确保所有依赖可用
2. **第 3️⃣ 部分** → 查看并根据需要调整 pipeline 配置
3. **第 4️⃣ 部分** → 运行 3DGS 快速训练演示（300 iterations）
4. **第 6️⃣ 部分** → 检查训练结果和关键指标

### 🔄 完整 Pipeline 执行
修改第 3️⃣ 部分的 `config["3dgs"]["iterations"] = 30000` 然后：
- 执行第 5️⃣ 部分（COLMAP，如需要）
- 执行第 7️⃣ 部分（调用 `run_full_pipeline(config)`）

### 📊 监控训练
- 查看第 8️⃣ 部分的日志文件更新
- 在终端运行 `tail -f logs/pipeline_*.log` 实时监控

### 📁 输出位置
- **训练模型**: `outputs/3dgs_demo/`
- **日志文件**: `logs/pipeline_*.log`
- **完整管道输出**: `outputs/`

---

**更新日期**: 2026-03-22  
**项目**: ME6402 3D Autonomous Retail  
**环境**: Conda (gaussian_splatting env)